In [0]:
# ============================================================
# NOTEBOOK : FACT_ORDER_ITEMS
# PURPOSE  : ORDER ITEMS FACT INCREMENTAL LOAD
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid


In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("order_items_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.fact_order_items"

    pipeline_name = "PL_FACT_ORDER_ITEMS"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            order_item_id,
            order_id,
            product_id,
            quantity,
            unit_price,
            discount_amount,
            total_amount,
            created_date,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_order_items"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : order_items_tbl
Rows Read : 5


Max Date updated successfully


FN_LOGGER LOADED SUCCESSFULLY


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
%sql
select * from config.watermark_table

table_name,last_load_timestamp,updated_timestamp
orders_tbl,2026-05-09T00:00:00.000Z,2026-05-27T09:09:26.392Z
inventory_tbl,2026-05-02T00:00:00.000Z,2026-05-27T08:14:43.205Z
product_tbl,2025-05-14T16:00:00.000Z,2026-05-27T07:54:23.602Z
supplier_tbl,2026-04-10T00:00:00.000Z,2026-05-27T07:43:17.460Z
customer_tbl,2025-05-25T12:00:00.000Z,2026-05-26T16:05:49.716Z
payments_tbl,1900-01-01T00:00:00.000Z,1900-01-01T00:00:00.000Z
category_tbl,2026-04-15T00:00:00.000Z,2026-05-27T07:25:22.101Z


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            oi.order_item_id,

            oi.order_id,

            o.customer_id,

            o.customer_name,

            oi.product_id,

            p.product_name,

            oi.quantity,

            oi.unit_price,

            (oi.quantity * oi.unit_price)
                AS gross_amount,

            oi.discount_amount,

            oi.total_amount,

            o.order_status,

            CASE

                WHEN oi.total_amount >= 50000
                THEN 'HIGH_VALUE_SALE'

                WHEN oi.total_amount >= 10000
                THEN 'MEDIUM_VALUE_SALE'

                ELSE 'LOW_VALUE_SALE'

            END AS sales_category,

            oi.modified_date,

            sha2(
                concat_ws(
                    '|',
                    oi.order_id,
                    oi.product_id,
                    oi.quantity,
                    oi.total_amount
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_order_items oi

        LEFT JOIN silver.dim_order o
            ON oi.order_id = o.order_id
            AND o.is_current = 1

        LEFT JOIN silver.dim_product p
            ON oi.product_id = p.product_id
            AND p.is_current = 1

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_order_items"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)



Silver View Created : order_items_tbl


In [0]:
# ============================================================
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE silver.fact_order_items
(
    order_item_id BIGINT,
    order_id BIGINT,
    customer_id BIGINT,
    customer_name STRING,
    product_id BIGINT,
    product_name STRING,
    quantity BIGINT,
    unit_price DOUBLE,
    gross_amount DOUBLE,
    discount_amount DOUBLE,
    total_amount DOUBLE,
    order_status STRING,
    sales_category STRING,
    modified_date TIMESTAMP,
    hash_key STRING
) 
USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.fact_order_items


In [0]:
try:
    spark.sql(""" TRUNCATE TABLE silver.fact_order_items """)
    
    print(f"Target Table Truncated : {table_name}")

except Exception as e:
    print(f"Target Table failed to Truncated : {table_name}")
    raise(e)


Target Table Truncated : order_items_tbl


In [0]:

try:
    spark.sql("""
              INSERT INTO silver.fact_order_items
               SELECT
                order_item_id,
                order_id,
                customer_id,
                customer_name,
                product_id,
                product_name,
                quantity,
                unit_price,
                gross_amount,
                discount_amount,
                total_amount,
                order_status,
                sales_category,
                modified_date,
                hash_key 
                FROM vw_silver_order_items 
                """)
    print(f"Data Inserted : {table_name}")
    
except Exception as e:
    print(f"Inserting to Table failed: {table_name}")
    raise(e)

Data Inserted : order_items_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"FACT_ORDER_ITEMS SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "FACT_ORDER_ITEMS",
        str(e)

    )

    print(f"FACT_ORDER_ITEMS LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : order_items_tbl
Watermark Updated : order_items_tbl
Audit Log Inserted : order_items_tbl
FACT_ORDER_ITEMS SUCCESSFULLY LOADED : order_items_tbl
